In [1]:
import duckdb

In [5]:
df = duckdb.sql("select* from 'venezuela_wdi_indicators.csv'").df()
df

,country_iso3,year,oil_rents_pct_gdp,total_natural_resource_rents_pct_gdp,fuel_exports_pct_merch_exports,ores_and_metals_exports_pct_merch_exports,gdp_current_usd,gdp_growth_pct
0,VEN,1960,NaN,NaN,NaN,NaN,7.663938e+09,NaN
1,VEN,1961,NaN,NaN,NaN,NaN,8.067267e+09,3.192519
2,VEN,1962,NaN,NaN,92.362928,0.180690,8.814310e+09,8.532934
3,VEN,1963,NaN,NaN,92.911190,3.766721,9.608717e+09,3.900951
4,VEN,1964,NaN,NaN,91.099545,5.871650,8.192414e+09,11.129345
...,...,...,...,...,...,...,...,...
60,VEN,2020,NaN,NaN,NaN,NaN,4.283802e+10,-29.998570
61,VEN,2021,NaN,NaN,NaN,NaN,5.661498e+10,0.955433
62,VEN,2022,NaN,NaN,NaN,NaN,8.901326e+10,8.000913
63,VEN,2023,NaN,NaN,NaN,NaN,1.023765e+11,4.001686


In [ ]:
duckdb.sql("describe select* from 'venezuela_wdi_indicators.csv'").df()

,column_name,column_type,null,key,default,extra
0,country_iso3,VARCHAR,YES,None,None,None
1,year,BIGINT,YES,None,None,None
2,oil_rents_pct_gdp,DOUBLE,YES,None,None,None
3,total_natural_resource_rents_pct_gdp,DOUBLE,YES,None,None,None
4,fuel_exports_pct_merch_exports,DOUBLE,YES,None,None,None
5,ores_and_metals_exports_pct_merch_exports,DOUBLE,YES,None,None,None
6,gdp_current_usd,DOUBLE,YES,None,None,None
7,gdp_growth_pct,DOUBLE,YES,None,None,None


1. Vue d'ensemble

In [14]:
duckdb.sql("select year,gdp_current_usd as PIB, gdp_growth_pct as croissance from 'venezuela_wdi_indicators.csv' where  croissance is not null order by year asc").df()

,year,PIB,croissance
0,1961,8.067267e+09,3.192519
1,1962,8.814310e+09,8.532934
2,1963,9.608717e+09,3.900951
3,1964,8.192414e+09,11.129345
4,1965,8.427778e+09,4.162867
...,...,...,...
59,2020,4.283802e+10,-29.998570
60,2021,5.661498e+10,0.955433
61,2022,8.901326e+10,8.000913
62,2023,1.023765e+11,4.001686


Les 5 pires années

In [19]:
duckdb.sql("select year,gdp_current_usd as PIB, gdp_growth_pct as croissance from 'venezuela_wdi_indicators.csv' where croissance is not null order by croissance asc limit 5").df()

,year,PIB,croissance
0,2020,4.283802e+10,-29.998570
1,2019,7.301408e+10,-27.657968
2,2018,1.020211e+11,-19.655342
3,2016,1.129150e+11,-17.040335
4,2017,1.158833e+11,-15.671409


Dépendance prétroliére par décénnie

In [41]:
duckdb.sql("select CAST(year/10 as integer)*10 as decade, round(avg(oil_rents_pct_gdp),2) from 'venezuela_wdi_indicators.csv' where oil_rents_pct_gdp is not null group by decade order by decade").df()

,decade,"round(avg(oil_rents_pct_gdp), 2)"
0,1970,12.38
1,1980,21.29
2,1990,15.68
3,2000,18.40
4,2010,17.99


4. Variation d'une année sur l'autre

In [42]:
duckdb.sql("select year, gdp_current_usd as PIB, lag(gdp_current_usd) over (order by year) as lag_PIB from 'venezuela_wdi_indicators.csv' where PIB is not null order by year").df()


,year,PIB,lag_PIB
0,1960,7.663938e+09,NaN
1,1961,8.067267e+09,7.663938e+09
2,1962,8.814310e+09,8.067267e+09
3,1963,9.608717e+09,8.814310e+09
4,1964,8.192414e+09,9.608717e+09
...,...,...,...
60,2020,4.283802e+10,7.301408e+10
61,2021,5.661498e+10,4.283802e+10
62,2022,8.901326e+10,5.661498e+10
63,2023,1.023765e+11,8.901326e+10


In [43]:
duckdb.sql("select year, gdp_current_usd as PIB, lag(gdp_current_usd) over (order by year) as lag_PIB, PIB - lag_PIB as Différence from 'venezuela_wdi_indicators.csv' where PIB is not null order by year").df()


,year,PIB,lag_PIB,Différence
0,1960,7.663938e+09,NaN,NaN
1,1961,8.067267e+09,7.663938e+09,4.033287e+08
2,1962,8.814310e+09,8.067267e+09,7.470429e+08
3,1963,9.608717e+09,8.814310e+09,7.944074e+08
4,1964,8.192414e+09,9.608717e+09,-1.416303e+09
...,...,...,...,...
60,2020,4.283802e+10,7.301408e+10,-3.017606e+10
61,2021,5.661498e+10,4.283802e+10,1.377696e+10
62,2022,8.901326e+10,5.661498e+10,3.239828e+10
63,2023,1.023765e+11,8.901326e+10,1.336327e+10


Corrélation pétrole/croissance (Pas de corrélation entre le pétrole et la croissance)

In [56]:
duckdb.sql("select corr(gdp_growth_pct, oil_rents_pct_gdp) as correlation from 'venezuela_wdi_indicators.csv' where oil_rents_pct_gdp is not null").df()

,correlation
0,0.048617


Moyenne Mobile 3 ans

In [60]:
duckdb.sql("select year, gdp_current_usd as PIB, avg(gdp_current_usd) over (order by year rows between 2 preceding and current row) as moving_avg from 'venezuela_wdi_indicators.csv'").df()

,year,PIB,moving_avg
0,1960,7.663938e+09,7.663938e+09
1,1961,8.067267e+09,7.865603e+09
2,1962,8.814310e+09,8.181838e+09
3,1963,9.608717e+09,8.830098e+09
4,1964,8.192414e+09,8.871814e+09
...,...,...,...
60,2020,4.283802e+10,7.262440e+10
61,2021,5.661498e+10,5.748903e+10
62,2022,8.901326e+10,6.282208e+10
63,2023,1.023765e+11,8.266825e+10
